In [1]:
import os

In [2]:
%pwd

'd:\\AI Projects\\Text_Summarization\\AI-Text-Summerizer-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\AI Projects\\Text_Summarization\\AI-Text-Summerizer-Project'

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int
    gradient_checkpointing : bool
    fp16: bool
    bf16: bool
    max_grad_norm: float
    use_cpu: bool

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps,
            gradient_checkpointing = params.gradient_checkpointing,
            fp16 = params.fp16,
            bf16 = params.bf16,
            max_grad_norm = params.max_grad_norm,
            use_cpu = params.use_cpu
        )

        return model_trainer_config

In [8]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

c:\Users\nikhi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_bart = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_bart)
        
        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, num_train_epochs=self.config.num_train_epochs, warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size, per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay, logging_steps=self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy, eval_steps=self.config.eval_steps, save_steps=1e6,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps, gradient_checkpointing = self.config.gradient_checkpointing,
            fp16 = self.config.fp16, bf16 = self.config.bf16, max_grad_norm = self.config.max_grad_norm, use_cpu = self.config.use_cpu
        ) 
        trainer = Trainer(model=model_bart, args=trainer_args,
                  processing_class=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["train"], 
                  eval_dataset=dataset_samsum_pt["validation"])
        
        trainer.train()

        ## Save model
        model_bart.save_pretrained(os.path.join(self.config.root_dir,"bart-samsum-model"))
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))

In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-06-11 22:27:50,272: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-11 22:27:50,272: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-11 22:27:50,272: INFO: common: created directory at: artifacts]
[2026-06-11 22:27:50,272: INFO: common: created directory at: artifacts/model_trainer]
[2026-06-11 22:27:50,740: INFO: _client: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-06-11 22:27:50,766: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/config.json "HTTP/1.1 200 OK"]
[2026-06-11 22:27:51,031: INFO: _client: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"]
[2026-06-11 22:27:51,270: INFO: _client: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/tokeni

[2026-06-11 22:27:52,381: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-06-11 22:27:52,400: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/config.json "HTTP/1.1 200 OK"]


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 12793.01it/s]


[2026-06-11 22:27:53,106: INFO: _client: HTTP Request: HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-06-11 22:27:53,120: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-cnn/37f520fa929c961707657b28798b30c003dd100b/generation_config.json "HTTP/1.1 200 OK"]


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
c:\Users\nikhi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 